# 09 — Residual Analysis

**Difficulty:** ⭐⭐⭐ &nbsp;|&nbsp; **Estimated time:** 2.5 hours  
**Week 5 — ML Fundamentals & Linear Regression**

---

## Why This Matters

Linear regression does not just produce predictions — it makes mathematical **guarantees** about those predictions:  
coefficients are unbiased, confidence intervals are valid, p-values are trustworthy.  
But those guarantees only hold when the **residuals** behave in a specific way.

Residual analysis is the diagnostic layer that tells you *whether your model's promises are real*.

---

## Real-World Analogy First

**The Doctor After Prescribing Medicine**

A doctor prescribes a treatment for a patient's fever.  
After the treatment, the doctor checks on *remaining symptoms* — the leftover illness the medicine did not fix.  
If the remaining symptoms are **random and small**, the medicine worked.  
If the remaining symptoms show a **pattern** (e.g., worse in the evening, worse in older patients),  
the doctor knows something systemic was missed.

**Residuals are the remaining symptoms after your model has done its job.**  
If they are random and well-behaved → the model has captured the signal.  
If they show a pattern → something systematic was left on the table.

---

## What Are Residuals?

$$e_i = y_i^{\text{true}} - \hat{y}_i^{\text{predicted}}$$

Each residual is the **leftover error** for a single data point after the model makes its prediction.

- Positive residual → the model **under-predicted** (actual price higher than forecast)
- Negative residual → the model **over-predicted** (actual price lower than forecast)
- Perfect model → all residuals = 0 (never achievable on real data)

**Three things we check in residuals:**
1. **Homoscedasticity** — constant variance across all predicted values
2. **Normality** — residuals follow a normal distribution
3. **Independence** — residuals are not correlated with each other

---

**Dataset used throughout:** California Housing (sklearn)  
**Target:** median house value (price in $100,000 units)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# statsmodels for Breusch-Pagan and Durbin-Watson
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
sns.set_palette('muted')
print('Libraries loaded successfully.')

In [ ]:
# ── Load California Housing Data ───────────────────────────────────────────
housing = fetch_california_housing(as_frame=True)

df = housing.frame.copy()
# Target: MedHouseVal is in units of $100,000
# We will work with raw units first, then log-transform to demonstrate fixes

print('Shape:', df.shape)
print(df.describe().round(2))
print('\nTarget column: MedHouseVal (median house value × $100,000)')

In [ ]:
# ── Fit a Baseline Linear Regression Model ─────────────────────────────────
# Features: all columns except the target
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']          # raw price units

# Stratified split to keep price distribution balanced across train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features — not required for OLS coefficients, but helps numerics
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit ONLY on training data
X_test_sc  = scaler.transform(X_test)        # apply same scale to test

model = LinearRegression()
model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)
residuals = y_test.values - y_pred            # e_i = y_true - y_hat

print(f'R² on test set : {model.score(X_test_sc, y_test):.4f}')
print(f'Mean residual  : {residuals.mean():.4f}  (should be ≈ 0)')
print(f'Std of residuals: {residuals.std():.4f}')

---

## Part 1 — Homoscedasticity (Constant Variance)

### What It Means

**Homoscedasticity** (from Greek: *homo* = same, *skedasis* = spread) means the residuals have **the same spread (variance) regardless of the predicted value**.

**Analogy:** Imagine using a tape measure to estimate the length of different objects.  
Whether you measure a 1 cm pencil or a 10 m wall, your measurement error should be about the same size.  
If your error grows proportionally with the object's size, that is **heteroscedasticity** — non-constant variance.

For house prices: when predicting cheap houses ($100k), you might be off by $20k.  
But for luxury houses ($2M), being off by $20k is unrealistically precise — errors tend to scale with the price.

### How to Check: Residuals vs Fitted Values Plot

- **x-axis:** fitted (predicted) values $\hat{y}$
- **y-axis:** residuals $e_i = y_i - \hat{y}_i$
- **What you want:** a random horizontal band around zero — no funnel, no curve
- **Red flag:** a funnel shape (wider spread at one end) = heteroscedasticity

### Formula for Breusch-Pagan Test

The Breusch-Pagan test regresses squared residuals on the fitted values.  
If the fitted values significantly predict the squared residuals, variance is not constant.

$$H_0: \text{residuals have constant variance (homoscedastic)}$$
$$H_1: \text{variance changes with fitted values (heteroscedastic)}$$

A **p-value < 0.05** → reject $H_0$ → heteroscedasticity detected.

In [ ]:
# ── Residuals vs Fitted Values Plot (Homoscedasticity Check) ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left panel: raw price model ────────────────────────────────────────────
ax = axes[0]
ax.scatter(y_pred, residuals, alpha=0.3, s=15, color='steelblue')
ax.axhline(0, color='red', linewidth=1.5, linestyle='--', label='Zero line')

# Smooth trend line to reveal any systematic pattern
# Use LOWESS (Locally Weighted Scatterplot Smoothing) from statsmodels
lowess_fit = sm.nonparametric.lowess(residuals, y_pred, frac=0.3)
ax.plot(lowess_fit[:, 0], lowess_fit[:, 1], color='red',
        linewidth=2, label='LOWESS trend')

ax.set_xlabel('Fitted Values ($100k)')
ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Fitted — RAW Price\n(Look for funnel shape)')
ax.legend()

# ── Right panel: log-transformed price model ───────────────────────────────
y_log = np.log(y)               # log-transform shrinks large price variance

X_train2, X_test2, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)
X_train_sc2 = scaler.fit_transform(X_train2)
X_test_sc2  = scaler.transform(X_test2)

model_log = LinearRegression()
model_log.fit(X_train_sc2, y_train_log)

y_pred_log = model_log.predict(X_test_sc2)
residuals_log = y_test_log.values - y_pred_log

ax2 = axes[1]
ax2.scatter(y_pred_log, residuals_log, alpha=0.3, s=15, color='seagreen')
ax2.axhline(0, color='red', linewidth=1.5, linestyle='--')
lowess_log = sm.nonparametric.lowess(residuals_log, y_pred_log, frac=0.3)
ax2.plot(lowess_log[:, 0], lowess_log[:, 1], color='red', linewidth=2,
         label='LOWESS trend')
ax2.set_xlabel('Fitted Values (log scale)')
ax2.set_ylabel('Residuals (log scale)')
ax2.set_title('Residuals vs Fitted — LOG Price\n(More uniform spread)')
ax2.legend()

plt.suptitle('Homoscedasticity Check: Raw vs Log-Transformed Target',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Breusch-Pagan Test (formal statistical test for heteroscedasticity) ─────
# The test regresses squared residuals on the fitted values.
# We need to add a constant (intercept) column for statsmodels conventions.

def breusch_pagan_report(fitted_values, resids, label='Model'):
    """Run Breusch-Pagan test and print a readable report."""
    # Add constant column so the BP regression has an intercept
    exog = sm.add_constant(fitted_values)
    bp_test = het_breuschpagan(resids, exog)

    lm_stat, lm_pval, f_stat, f_pval = bp_test
    print(f'\n─── Breusch-Pagan Test: {label} ───')
    print(f'  LM statistic : {lm_stat:.4f}')
    print(f'  LM p-value   : {lm_pval:.4f}')
    verdict = 'HETEROSCEDASTIC (p < 0.05)' if lm_pval < 0.05 else 'Homoscedastic (p ≥ 0.05)'
    print(f'  Verdict      : {verdict}')

breusch_pagan_report(y_pred,     residuals,     label='Raw price')
breusch_pagan_report(y_pred_log, residuals_log, label='Log price')

### Interpretation

| Model | Breusch-Pagan p-value | Verdict |
|-------|-----------------------|---------|
| Raw price | Very small (< 0.001) | Heteroscedastic — errors grow with price |
| Log price | Larger | More constant variance |

**Why does log-transforming the target help?**  
House prices span a huge range (\$50k to \$5M+). Errors on expensive houses are naturally larger in dollar terms.  
Taking the log compresses the scale so a $\\$100k$ error on a $\\$1M$ house becomes the same scale as a $\\$10k$ error on a $\\$100k$ house.

**Fix options:**
- `log(y)` — most common for price data
- `sqrt(y)` — for count data
- **Weighted Least Squares (WLS):** give smaller weights to observations with higher expected variance

---

## Part 2 — Normality of Residuals

### What It Means

Residuals should follow a **normal (Gaussian) distribution** centered at zero.

**Analogy:** Your prediction errors should behave like random noise from a radio — equally likely to be a little too high or too low, very rarely extreme, and following the familiar bell curve shape.

### Why It Matters

- The **t-tests** on coefficients (checking if a feature is significant) assume residuals are normal
- **Confidence intervals** for predictions rely on normality
- For large samples (n > 100), the Central Limit Theorem partially rescues us — but we should still check

### How to Check

**1. Histogram of residuals** — should look bell-shaped  
**2. Q-Q Plot (Quantile-Quantile Plot)**
  - Plots the quantiles of your residuals against the quantiles of a theoretical normal distribution
  - If residuals are normal → points lie on a 45° line
  - Deviations from the line reveal skewness (one tail curves away) or heavy tails (both ends curve away)

### Shapiro-Wilk Test

$$H_0: \text{residuals are normally distributed}$$
$$H_1: \text{residuals are NOT normally distributed}$$

**p-value < 0.05** → reject normality  
Note: with very large samples, Shapiro-Wilk almost always rejects even small, harmless deviations — use your eyes too.

In [ ]:
# ── Normality Check: Histogram + Q-Q Plot ─────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ─ Histogram: Raw residuals ─────────────────────────────────────────────────
ax = axes[0, 0]
ax.hist(residuals, bins=60, density=True, color='steelblue',
        alpha=0.7, edgecolor='white', label='Residuals')
# Overlay a fitted normal curve for visual comparison
x_grid = np.linspace(residuals.min(), residuals.max(), 300)
normal_curve = stats.norm.pdf(x_grid, residuals.mean(), residuals.std())
ax.plot(x_grid, normal_curve, 'r-', linewidth=2, label='Normal fit')
ax.set_title('Histogram of Residuals\n(Raw price model)')
ax.set_xlabel('Residual value')
ax.set_ylabel('Density')
ax.legend()

# ─ Q-Q Plot: Raw residuals ──────────────────────────────────────────────────
ax = axes[0, 1]
# probplot computes theoretical quantiles and orders the residuals
(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
ax.scatter(osm, osr, alpha=0.3, s=10, color='steelblue', label='Residuals')
ax.plot(osm, slope * np.array(osm) + intercept, 'r-',
        linewidth=2, label='Normal reference line')
ax.set_title('Q-Q Plot — Raw Residuals\n(Deviations from line = non-normality)')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.legend()

# ─ Histogram: Log-transformed residuals ────────────────────────────────────
ax = axes[1, 0]
ax.hist(residuals_log, bins=60, density=True, color='seagreen',
        alpha=0.7, edgecolor='white', label='Residuals (log price)')
x_grid2 = np.linspace(residuals_log.min(), residuals_log.max(), 300)
normal_curve2 = stats.norm.pdf(x_grid2, residuals_log.mean(), residuals_log.std())
ax.plot(x_grid2, normal_curve2, 'r-', linewidth=2, label='Normal fit')
ax.set_title('Histogram of Residuals\n(Log price model)')
ax.set_xlabel('Residual value (log scale)')
ax.set_ylabel('Density')
ax.legend()

# ─ Q-Q Plot: Log residuals ──────────────────────────────────────────────────
ax = axes[1, 1]
(osm2, osr2), (slope2, intercept2, r2) = stats.probplot(residuals_log, dist='norm')
ax.scatter(osm2, osr2, alpha=0.3, s=10, color='seagreen')
ax.plot(osm2, slope2 * np.array(osm2) + intercept2, 'r-', linewidth=2,
        label='Normal reference line')
ax.set_title('Q-Q Plot — Log Residuals\n(Much closer to the line)')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.legend()

plt.suptitle('Normality of Residuals: Raw vs Log-Transformed Price',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Shapiro-Wilk Normality Test ────────────────────────────────────────────
# Shapiro-Wilk is reliable for n < ~5000; for larger samples use KS test
# We sample 2000 points to keep Shapiro-Wilk in its valid range

rng = np.random.default_rng(42)
sample_idx = rng.choice(len(residuals), size=2000, replace=False)

stat_raw, p_raw = stats.shapiro(residuals[sample_idx])
stat_log, p_log = stats.shapiro(residuals_log[sample_idx])

print('─── Shapiro-Wilk Normality Test (sample n=2000) ───')
print(f'  Raw price residuals  → W={stat_raw:.4f}, p={p_raw:.6f}')
print(f'  Log price residuals  → W={stat_log:.4f}, p={p_log:.6f}')

# Kolmogorov-Smirnov test (also works for large n)
ks_raw  = stats.kstest(residuals,     'norm',
                       args=(residuals.mean(), residuals.std()))
ks_log  = stats.kstest(residuals_log, 'norm',
                       args=(residuals_log.mean(), residuals_log.std()))
print('\n─── Kolmogorov-Smirnov Test (all test points) ───')
print(f'  Raw price residuals  → KS={ks_raw.statistic:.4f}, p={ks_raw.pvalue:.6f}')
print(f'  Log price residuals  → KS={ks_log.statistic:.4f}, p={ks_log.pvalue:.6f}')

print('\nSkewness of residuals:')
print(f'  Raw : {stats.skew(residuals):.3f}  (0 = perfectly symmetric)')
print(f'  Log : {stats.skew(residuals_log):.3f}')

### Reading the Q-Q Plot

| Pattern in Q-Q Plot | Meaning |
|---------------------|---------|
| Points on the line | Residuals are normally distributed |
| Right tail curves up | Right-skewed (positive skew) — large positive errors are bigger than expected |
| Left tail curves down | Left-skewed — large negative errors are bigger than expected |
| Both tails curve away from line | Heavy-tailed distribution (kurtosis > 3) |
| S-shape | Non-linearity in the relationship |

**For house prices:** raw prices produce right-skewed residuals (expensive houses are rarer but more variable).  
Log-transforming pulls the right tail in, making residuals much closer to normal.

**Fix options:**
- Log or square-root transformation of the target
- Remove extreme outliers (carefully — understand why they are extreme)
- Use **robust regression** (Huber loss) — less sensitive to non-normal tails

---

## Part 3 — Independence of Residuals

### What It Means

Residuals should be **independent of each other** — knowing one residual should tell you nothing about the next.

**Analogy:** Imagine predicting house prices month by month.  
If your model under-predicts in January, February, and March (all in a row),  
your errors are **not independent** — they cluster in time.  
This is called **autocorrelation** (errors are correlated with their own lagged values).

### When Does This Happen?

- **Time series data:** consecutive rows are neighboring time periods
- **Spatial data:** nearby houses are priced similarly → prediction errors cluster by location
- **Panel data:** repeated measurements from the same house

### Durbin-Watson Statistic

$$DW = \frac{\sum_{i=2}^{n}(e_i - e_{i-1})^2}{\sum_{i=1}^{n}e_i^2}$$

- **DW ≈ 2:** no autocorrelation (independent)
- **DW < 2:** positive autocorrelation (similar errors repeat)
- **DW > 2:** negative autocorrelation (errors oscillate sign)
- **Rule of thumb:** values between 1.5 and 2.5 are generally acceptable

In [ ]:
# ── Simulating Autocorrelated Residuals (time-ordered data) ────────────────
# California housing is cross-sectional (not time-ordered), so we will
# simulate a time-ordered scenario to demonstrate autocorrelation.

np.random.seed(42)
n_houses = 300
time_index = np.arange(n_houses)

# Independent residuals (what we want)
resid_independent = np.random.normal(0, 1, n_houses)

# Autocorrelated residuals: each error = 0.8 × previous error + small noise
# AR(1) process: e_t = phi * e_{t-1} + noise
phi = 0.85                         # high autocorrelation coefficient
resid_autocorr = np.zeros(n_houses)
resid_autocorr[0] = np.random.normal(0, 1)
for t in range(1, n_houses):
    resid_autocorr[t] = phi * resid_autocorr[t-1] + np.random.normal(0, 0.5)

# ── Plot both ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(time_index, resid_independent, color='steelblue',
             alpha=0.7, linewidth=0.8)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].scatter(time_index, resid_independent, s=8, color='steelblue', alpha=0.5)
axes[0].set_title(f'Independent Residuals\nDW = {durbin_watson(resid_independent):.2f} (target: ≈ 2)')
axes[0].set_xlabel('Row / Time Index')
axes[0].set_ylabel('Residual')

axes[1].plot(time_index, resid_autocorr, color='crimson',
             alpha=0.7, linewidth=0.8)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].scatter(time_index, resid_autocorr, s=8, color='crimson', alpha=0.5)
axes[1].set_title(f'Autocorrelated Residuals (φ={phi})\nDW = {durbin_watson(resid_autocorr):.2f} (far from 2 = bad)')
axes[1].set_xlabel('Row / Time Index')
axes[1].set_ylabel('Residual')

plt.suptitle('Independence Check: Residuals vs Time Index',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Lag-1 autocorrelation: correlation between e_t and e_{t-1}
lag1_indep   = np.corrcoef(resid_independent[:-1], resid_independent[1:])[0, 1]
lag1_autocorr = np.corrcoef(resid_autocorr[:-1],   resid_autocorr[1:])[0, 1]
print(f'Lag-1 autocorrelation — Independent: {lag1_indep:.3f}')
print(f'Lag-1 autocorrelation — Autocorrelated: {lag1_autocorr:.3f}')

In [ ]:
# ── Durbin-Watson on Real Model Residuals ──────────────────────────────────
# California housing rows are not time-ordered, so DW should be ≈ 2

dw_raw = durbin_watson(residuals)
dw_log = durbin_watson(residuals_log)

print('─── Durbin-Watson Statistic ───')
print(f'  Raw price model : DW = {dw_raw:.4f}')
print(f'  Log price model : DW = {dw_log:.4f}')
print('  (Values between 1.5–2.5 suggest no serious autocorrelation)')

# Fix for autocorrelated residuals:
print('\n─── Fixes for autocorrelation ───')
print('  1. Add lagged features (e.g., last month price, last month error)')
print('  2. Use time-series models: ARIMA, SARIMA, or GLS with AR errors')
print('  3. Include neighborhood/district as a categorical feature (spatial)')

---

## Part 4 — Linearity Check via Residuals vs Features

If your model is missing a **non-linear relationship**, it will show up in residuals vs feature plots.  
A curved pattern means the model is systematically wrong at different feature values — a polynomial term is needed.

**Analogy:** If you tried to fit a straight line through a parabolic road, the car would veer off in the curves.  
The residuals would be negative at the start, positive in the middle, negative at the end — a clear U-shape.

**Formula for detecting non-linearity:**  
There is no single test — you examine the pattern. A LOWESS smoother through the residuals vs feature plot reveals curvature.

In [ ]:
# ── Residuals vs Key Features (Linearity Check) ────────────────────────────
key_features = ['MedInc', 'HouseAge', 'AveRooms', 'Latitude']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, feat in zip(axes, key_features):
    feat_values = X_test[feat].values
    ax.scatter(feat_values, residuals, alpha=0.25, s=10, color='steelblue')
    ax.axhline(0, color='red', linestyle='--', linewidth=1.2)

    # LOWESS smoother to detect any systematic curve
    sort_idx = np.argsort(feat_values)
    lowess_out = sm.nonparametric.lowess(
        residuals[sort_idx], feat_values[sort_idx], frac=0.4
    )
    ax.plot(lowess_out[:, 0], lowess_out[:, 1], 'r-', linewidth=2)

    ax.set_xlabel(feat)
    ax.set_ylabel('Residual')
    ax.set_title(f'Residuals vs {feat}')

plt.suptitle('Linearity Check: Residuals vs Individual Features\n'
             '(Curve in LOWESS = missing polynomial term)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('If the red LOWESS line deviates noticeably from zero,'
      ' that feature needs a polynomial or log term.')

---

## Part 5 — Outliers, Leverage, and Influential Points

Not all data points have equal influence on the fitted model. Three types of unusual points:

| Type | X unusual? | y unusual? | Effect |
|------|-----------|-----------|--------|
| **Outlier** | No | Yes | Large residual — pulls the line toward it |
| **High-leverage point** | Yes | Maybe | Unusual X — like a long lever arm, can distort the slope |
| **Influential point** | Yes | Yes | Changing it significantly changes the model |

### Cook's Distance

Cook's Distance measures how much the fitted values change if we remove one data point:

$$D_i = \frac{\sum_{j=1}^{n}(\hat{y}_j - \hat{y}_{j(i)})^2}{p \cdot MSE}$$

Where $\hat{y}_{j(i)}$ is the predicted value for point $j$ when observation $i$ is removed.  
$p$ = number of predictors, $MSE$ = mean squared error.

**Rule of thumb:**  
- $D_i > 1$: potentially influential  
- $D_i > 4/n$: worth investigating

### Leverage (Hat Values)

The **hat matrix** $H = X(X^TX)^{-1}X^T$ tells us how much each observation "controls" its own fitted value.  
High leverage: $h_{ii} > 2p/n$ (unusual in the feature space)

In [ ]:
# ── Cook's Distance & Leverage Computation ─────────────────────────────────
# Use statsmodels OLS to get influence statistics conveniently
# We use a subset of test data for speed (full set would be slow)

# Use 500 test points for the influence analysis
n_subset = 500
rng = np.random.default_rng(0)
subset_idx = rng.choice(len(y_test), size=n_subset, replace=False)

X_sub = X_test_sc[subset_idx]
y_sub = y_test.values[subset_idx]

# Fit statsmodels OLS (same as sklearn LinearRegression but with diagnostics)
X_sub_const = sm.add_constant(X_sub)            # statsmodels needs explicit intercept
sm_model    = sm.OLS(y_sub, X_sub_const).fit()

# Extract influence measures
influence = sm_model.get_influence()
cooks_d   = influence.cooks_distance[0]          # Cook's D for each observation
leverage  = influence.hat_matrix_diag            # hat values h_ii
std_resid = influence.resid_studentized_external  # studentized residuals

# Threshold for Cook's D: 4/n is a commonly cited rule of thumb
cook_threshold = 4 / n_subset

top5_influential = np.argsort(cooks_d)[::-1][:5]  # top 5 by Cook's D
print('Top 5 most influential points (Cook\'s D):')
for rank, idx in enumerate(top5_influential):
    print(f'  #{rank+1}: point {idx:3d} | '
          f'Cook D={cooks_d[idx]:.4f} | '
          f'Leverage={leverage[idx]:.4f} | '
          f'Std resid={std_resid[idx]:.3f}')
print(f'\nCook\'s D threshold (4/n): {cook_threshold:.4f}')
print(f'Points exceeding threshold: {(cooks_d > cook_threshold).sum()}')

In [ ]:
# ── Cook's Distance Bar Plot + Influence (Leverage vs Residual) Plot ────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Left: Cook's Distance for all points ───────────────────────────────────
ax = axes[0]
colors = ['crimson' if d > cook_threshold else 'steelblue' for d in cooks_d]
ax.bar(range(n_subset), cooks_d, color=colors, width=1.0, alpha=0.7)
ax.axhline(cook_threshold, color='red', linestyle='--', linewidth=1.5,
           label=f'Threshold (4/n = {cook_threshold:.3f})')

# Annotate top 3 influential points
for idx in top5_influential[:3]:
    ax.annotate(f'#{idx}', (idx, cooks_d[idx]),
                textcoords='offset points', xytext=(3, 3), fontsize=8, color='darkred')

ax.set_xlabel('Observation Index')
ax.set_ylabel("Cook's Distance")
ax.set_title("Cook's Distance\n(Red bars exceed 4/n threshold)")
ax.legend()

# ── Right: Influence plot — leverage vs studentized residual ────────────────
ax = axes[1]
# Color points by Cook's D to show which combine high residual + high leverage
sc = ax.scatter(leverage, std_resid, c=cooks_d, cmap='YlOrRd',
                s=20, alpha=0.8, edgecolors='none')
plt.colorbar(sc, ax=ax, label="Cook's Distance")

# Reference lines: common leverage threshold = 2p/n
n_features_p1 = X_sub_const.shape[1]           # number of parameters (p+1)
leverage_threshold = 2 * n_features_p1 / n_subset
ax.axvline(leverage_threshold, color='blue', linestyle='--', linewidth=1.2,
           label=f'Leverage threshold (2p/n={leverage_threshold:.3f})')
ax.axhline(2, color='orange', linestyle='--', linewidth=1.2,
           label='Std residual = ±2')
ax.axhline(-2, color='orange', linestyle='--', linewidth=1.2)

ax.set_xlabel('Leverage (hat value hᵢᵢ)')
ax.set_ylabel('Studentized Residual')
ax.set_title('Influence Plot: Leverage vs Residual\n'
             '(Top-right quadrant = most influential)')
ax.legend(fontsize=8)

plt.suptitle('Outlier & Influence Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 6 — Full 4-Panel Diagnostic Plot

In R, the `plot(lm_model)` command produces four standard diagnostic plots.  
We recreate these in Python — this is the standard residual analysis dashboard used in practice.

| Panel | Plot | What to look for |
|-------|------|------------------|
| 1 | Residuals vs Fitted | No pattern, horizontal band around 0 |
| 2 | Q-Q plot | Points on the diagonal line |
| 3 | Scale-Location (√\|residuals\| vs fitted) | Horizontal band — tests homoscedasticity |
| 4 | Residuals vs Leverage | No points in top-right or bottom-right corners |

In [ ]:
# ── 4-Panel Regression Diagnostic Plot (R-style) ───────────────────────────
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig)

sm_resid   = sm_model.resid
sm_fitted  = sm_model.fittedvalues
sqrt_abs_resid = np.sqrt(np.abs(sm_resid))

# ── Panel 1: Residuals vs Fitted ───────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(sm_fitted, sm_resid, alpha=0.35, s=12, color='steelblue')
ax1.axhline(0, color='red', linestyle='--', linewidth=1.5)
lw1 = sm.nonparametric.lowess(sm_resid, sm_fitted, frac=0.4)
ax1.plot(lw1[:, 0], lw1[:, 1], 'r-', linewidth=2)
ax1.set_xlabel('Fitted Values')
ax1.set_ylabel('Residuals')
ax1.set_title('1 — Residuals vs Fitted')

# ── Panel 2: Normal Q-Q ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
(qq_x, qq_y), (slope_q, int_q, _) = stats.probplot(sm_resid, dist='norm')
ax2.scatter(qq_x, qq_y, alpha=0.35, s=12, color='seagreen')
ax2.plot(qq_x, np.array(qq_x) * slope_q + int_q, 'r-', linewidth=2)
ax2.set_xlabel('Theoretical Quantiles')
ax2.set_ylabel('Sample Quantiles')
ax2.set_title('2 — Normal Q-Q')

# ── Panel 3: Scale-Location ────────────────────────────────────────────────
# Uses √|standardized residuals| to amplify variance pattern
ax3 = fig.add_subplot(gs[1, 0])
ax3.scatter(sm_fitted, sqrt_abs_resid, alpha=0.35, s=12, color='darkorange')
lw3 = sm.nonparametric.lowess(sqrt_abs_resid, sm_fitted, frac=0.4)
ax3.plot(lw3[:, 0], lw3[:, 1], 'r-', linewidth=2)
ax3.set_xlabel('Fitted Values')
ax3.set_ylabel('√|Standardized Residuals|')
ax3.set_title('3 — Scale-Location (Homoscedasticity)')

# ── Panel 4: Residuals vs Leverage ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.scatter(leverage, std_resid, alpha=0.4, s=12, color='mediumpurple')
ax4.axhline(0, color='gray', linestyle='--')
ax4.axvline(leverage_threshold, color='blue', linestyle='--',
            linewidth=1, label='Leverage threshold')

# Mark top influential points
for idx in top5_influential[:3]:
    ax4.annotate(f'#{idx}', (leverage[idx], std_resid[idx]),
                 fontsize=8, color='darkred',
                 textcoords='offset points', xytext=(4, 2))

ax4.set_xlabel('Leverage')
ax4.set_ylabel('Studentized Residuals')
ax4.set_title('4 — Residuals vs Leverage')
ax4.legend(fontsize=8)

plt.suptitle('4-Panel Regression Diagnostic Dashboard\n(Modelled on R\'s plot.lm)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Summary: What to Look For and How to Fix It

| Assumption | Diagnostic Plot | Statistical Test | Common Fix |
|------------|----------------|------------------|------------|
| Homoscedasticity | Residuals vs Fitted (fan shape = bad) | Breusch-Pagan | Log-transform target, WLS |
| Normality | Q-Q plot, histogram | Shapiro-Wilk, KS | Log-transform, remove outliers |
| Independence | Residuals vs time/index | Durbin-Watson | Lagged features, GLS, ARIMA |
| Linearity | Residuals vs each feature | LOWESS curve | Polynomial terms, interaction terms |
| No outliers | Cook's distance, leverage plot | Visual + threshold | Investigate & remove if justified |

---

## Self-Check Questions

<details>
<summary><strong>Q1: Your residuals vs fitted values plot shows a "fan" shape — residuals spread wider for higher predicted values. What assumption is violated and how would you fix it?</strong></summary>

**Violated assumption:** Homoscedasticity (constant variance). This is **heteroscedasticity** — the variance of errors increases with the predicted value.  

**Fixes:**
1. **Log-transform the target:** `log(price)` normalizes the scale, pulling in the large errors for expensive houses
2. **Square-root transform:** milder compression — good for count data
3. **Weighted Least Squares (WLS):** assign smaller weights to observations with larger expected variance
4. **Confirm with Breusch-Pagan test:** a p-value < 0.05 formally confirms heteroscedasticity

The fan shape is extremely common with price data because percentage errors are more natural than absolute errors for prices.

</details>

<details>
<summary><strong>Q2: The Q-Q plot shows residuals curving upward at the right tail. What does this indicate about the distribution of errors?</strong></summary>

The right tail curving **above** the reference line means the largest positive residuals are **bigger than a normal distribution would predict** — the distribution has a **heavy right tail** (positive skew).  

In house price terms: the model occasionally *severely* under-predicts expensive houses (actual price much higher than predicted). These extreme positive errors are more common than a normal distribution assumes.  

**Fix:** log-transform the target to compress the right tail. After transformation, the Q-Q plot should tighten to the diagonal.

</details>

<details>
<summary><strong>Q3: Cook's distance for house #1247 is 2.3 (all others are < 0.1). What does this mean and what should you do?</strong></summary>

**Interpretation:** House #1247 is **highly influential** — removing it from the dataset would substantially change the fitted coefficients. Cook's D = 2.3 far exceeds the common threshold of 4/n (which would be ≈ 0.002 for n=2000).  

**What to do:**
1. **Investigate the house:** Is it a data entry error? A truly unique property? A celebrity mansion?
2. **Fit the model without it:** Compare the coefficients. If they change drastically, the point is a problem.
3. **Do NOT blindly delete it:** If it is a valid data point, removing it introduces selection bias. Instead, use **robust regression** (Huber loss) which automatically down-weights influential points.
4. **Check for data errors:** A price of $1 for a house or 500 bedrooms are recording errors — fix or remove.

</details>

<details>
<summary><strong>Q4: You are predicting daily house prices over time. Your residuals vs time plot shows a wave pattern. What assumption is violated?</strong></summary>

**Violated assumption:** Independence of residuals. The wave pattern indicates **positive autocorrelation** — if the model over-predicts one day, it tends to over-predict the next day too.  

**Durbin-Watson statistic** would be significantly less than 2 (possibly around 0.5–1.2).  

**Why it happens:** Real estate markets have momentum — prices trend up or down together across consecutive days, driven by interest rates, economic news, or seasonal demand.  

**Fixes:**
1. **Add lagged features:** `price_yesterday`, `rolling_7day_avg_price`
2. **Include time-based features:** month-of-year, quarter, interest rate at that time
3. **Switch to a time-series model:** ARIMA, SARIMA, or a regression with AR errors (GLS)
4. **Consequence if ignored:** Standard errors are under-estimated → t-statistics are inflated → you will wrongly conclude features are significant when they are not

</details>